<span style="color: #808080;">_I'm hoping that people reading this notebook have already seen the poster and have a general idea about the **why** behind biased splitting. The goal of these structured notebooks is to show (to the best of my abilities) **how** the idea came to be and how it was executed._</span>

 # Downloading the Datasets

I'm using the dataset curated by [Landrum & Riniker](https://pubs.acs.org/doi/10.1021/acs.jcim.4c00049) where they show **Combining IC50 or Ki Values from Different Sources Is a Source of Significant Noise**.

The dataset can be downloaded from [here](https://github.com/rinikerlab/overlapping_assays).

In [1]:
%%bash

rm -rf ../tmpdata/ ../data/
git clone https://github.com/rinikerlab/overlapping_assays.git ../tmpdata/
mkdir -p ../data/raw
gzip -d ../tmpdata/datasets/source_data/*IC50.csv.gz
mv ../tmpdata/datasets/source_data/*IC50.csv ../data/raw/
rm -rf ../tmpdata

Cloning into '../tmpdata'...


## Standardizing SMILES
Standardize SMILES and report Canonical SMILES, Remove Failed SMILES, Deduplicate them, and Save the results in benchmark/data/standardized

In [2]:
import os

import molvs
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import rdchem, rdMolDescriptors as rdmd
from tqdm.notebook import tqdm

In [3]:
RDLogger.DisableLog("rdApp.*")

In [4]:
os.makedirs('../data/standardized/', exist_ok=True)

## Note on Standardization of SMILES
Standardization is based on checking if the input SMILES (in the dataset) is valid, and then disconnecting the metal(/s), Choosing the largest organic fragment (here, organic means the fragment which has at least one carbon) and finally un-charging the molecule. This molecule then, if valid, is converted into canonical SMILES and returned as a string. If there is any error in this process, None is returned.

In [5]:
md = molvs.metal.MetalDisconnector()
lfc = molvs.fragment.LargestFragmentChooser()
uc = molvs.charge.Uncharger()

In [6]:
def standardize_smiles(smiles):
    std_smiles = molvs.standardize.standardize_smiles(smiles)
    std_mol = Chem.MolFromSmiles(std_smiles)
    disconnected_std_mol = md.disconnect(std_mol)
    largest_organic_fragment_std_mol = lfc.choose(disconnected_std_mol)
    uncharted_std_mol = uc.uncharge(largest_organic_fragment_std_mol)
    std_smi = Chem.MolToSmiles(uncharted_std_mol)
    if not molvs.validate.validate_smiles(std_smi):
        canonical_tautomer_std_smi = molvs.standardize.canonicalize_tautomer_smiles(std_smi) # Very Slow
        return canonical_tautomer_std_smi
    return None

In [7]:
for file_name in tqdm(os.listdir('../data/raw/'), desc="Standardize", leave=True, position=0):
    if file_name.endswith('.csv'):
        df = pd.read_csv(f'../data/raw/{file_name}')
        df["standardized_smiles"] = df["canonical_smiles"].apply(standardize_smiles)
        df.dropna(subset=["standardized_smiles"], inplace=True)
        df.drop_duplicates(subset=["standardized_smiles"], inplace=True)
        df.to_csv(f'../data/standardized/{file_name}', index=False)

Standardize:   0%|          | 0/67 [00:00<?, ?it/s]